# 02 — Train Lead Adapter (GRPO)

**Purpose:** Fine-tune the Lead LoRA adapter using GRPO on rollout data collected from the live env. Upload trained adapter to HF Hub.

**Runtime:** Colab T4 · ~3h

> Requires notebook 01 to have run in the same session (or re-run cells 1-4 here).

In [ ]:
# Cell 1 — Install (skip if already done in 01)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.13" peft accelerate bitsandbytes
!pip install -q "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv@v0.2.3"
!pip install -q wandb huggingface_hub datasets
import sys; sys.path.insert(0, 'sdk_sovereign_pkg')
print('✓ Ready')

In [ ]:
# Cell 2 — Config
HF_USER  = 'ishansurdi'
ENV_URL  = 'https://ishansurdi-sdk-sovereign.hf.space'
N_ROLLOUT_EPISODES = 80
WANDB_PROJECT = 'sdk-sovereign'

In [ ]:
# Cell 3 — Load model + agents (re-run if kernel is fresh)
from server.llm_agents import load_model_with_two_adapters, make_agent_pair
from google.colab import userdata
from huggingface_hub import login
import wandb, os

login(token=userdata.get('HF_TOKEN'))
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY', '')

model, tokenizer = load_model_with_two_adapters()
agents = make_agent_pair(model, tokenizer)
print('✓ Model + agents loaded')

In [ ]:
# Cell A — Collect rollouts
import json
from pathlib import Path
from client import SDKSovereignEnv
from models import SDKAction

rollout_data = {'auditor': [], 'lead': []}

for ep in range(N_ROLLOUT_EPISODES):
    with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
        obs = env.reset()
        per_role_buffer = []
        while not obs.done and obs.turn_index < 7:
            agent = agents[obs.current_role]
            agent.model.set_adapter(agent.adapter_name)
            prompt = agent._build_prompt(obs)
            completion = agent._generate(prompt)
            action = agent._parse_action(completion)
            new_obs = env.step(action)
            per_role_buffer.append({
                'role': obs.current_role,
                'prompt': prompt,
                'completion': completion,
                'step_reward': new_obs.reward,
            })
            obs = new_obs
        for entry in per_role_buffer:
            rollout_data[entry['role']].append({
                'prompt': entry['prompt'],
                'reward': entry['step_reward'],
            })
    if ep % 10 == 0:
        print(f'  rollout {ep}/{N_ROLLOUT_EPISODES}')

print(f'Lead samples:    {len(rollout_data["lead"])}')
print(f'Auditor samples: {len(rollout_data["auditor"])}')
Path('rollout_lead.jsonl').write_text('\n'.join(json.dumps(r) for r in rollout_data['lead']))
Path('rollout_auditor.jsonl').write_text('\n'.join(json.dumps(r) for r in rollout_data['auditor']))
print('✓ Rollout data saved')

In [ ]:
# Cell B — GRPO train Lead adapter
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

wandb.init(project=WANDB_PROJECT, name='lead-grpo-round1')

lead_prompts = [r['prompt'] for r in rollout_data['lead']]
ds_lead = Dataset.from_dict({'prompt': lead_prompts})

def lead_reward_fn(completions, **kwargs):
    rewards = []
    for completion in completions:
        action = agents['lead']._parse_action(completion)
        if action.action_type == 'submit_patch':
            with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                obs = env.reset()
                env.step(SDKAction(role='auditor', action_type='pass'))
                env.step(SDKAction(role='lead', action_type='propose_replacement',
                                   proposed_sdk='razorpay'))
                env.step(SDKAction(role='auditor', action_type='approve'))
                final = env.step(action)
                rewards.append(float(final.reward))
        elif action.action_type == 'propose_replacement':
            rewards.append(1.0)
        elif action.action_type == 'request_hint':
            rewards.append(0.3)
        else:
            rewards.append(-0.5)  # pass
    return rewards

config = GRPOConfig(
    output_dir='checkpoints/lead',
    num_generations=4,
    max_completion_length=512,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=2,
    save_steps=50,
    report_to='wandb',
)

model.set_adapter('lead_adapter')
for n, p in model.named_parameters():
    if 'auditor_adapter' in n:
        p.requires_grad = False

trainer = GRPOTrainer(
    model=model,
    reward_funcs=lead_reward_fn,
    args=config,
    train_dataset=ds_lead.select(range(min(60, len(ds_lead)))),
    tokenizer=tokenizer,
)
trainer.train()
wandb.finish()
print('✓ Lead GRPO training complete')

In [ ]:
# Cell C — Save and push Lead adapter to HF Hub
from huggingface_hub import HfApi

model.save_pretrained('checkpoints/lead_adapter_v1', selected_adapters=['lead_adapter'])
HfApi().upload_folder(
    folder_path='checkpoints/lead_adapter_v1',
    repo_id=f'{HF_USER}/sdk-sovereign-lead-adapter',
    repo_type='model',
    commit_message='Lead LoRA adapter v1 (GRPO round 1)',
)
print(f'✓ Lead adapter pushed to HF Hub: {HF_USER}/sdk-sovereign-lead-adapter')